In [1]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GITHUB_PAT")
!rm -rf /kaggle/working/*
!git clone -q https://{tok}@github.com/vhrabar/SolarFilamentSegmentationChallenge.git

In [2]:
!cd SolarFilamentSegmentationChallenge && git checkout dev && git pull

Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'
Already up to date.


## Train


In [ ]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --seed 1 --device cuda:0 train \
    --arch transunet --crop 512 --crops-per-image 4 --foreground-prob 0.7 --pos-weight 5 \
    --batch-size 8 --epochs 40 --lr 2e-4 --dropout 0.1 --drop-path 0.1 --warmup-epochs 3 \
    --amp --workers 4 --swa-epochs 12 \
    --val-limit 64 --val-tile 512 --val-overlap 0.0 --tile-batch 8 \
    --output-dir /kaggle/working/runs/transunet_crop512_pw5

## Calibrate



In [ ]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --device cuda:0 calibrate \
    --checkpoint /kaggle/working/runs/transunet_crop512_pw5/swa.pt --tile 512 --overlap 0.25 --tile-batch 8 --workers 2

## Predict

Threshold, offset and minimum area are read back out of the checkpoint.

In [ ]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --device cuda:0 predict \
    --checkpoint /kaggle/working/runs/transunet_crop512_pw5/swa.pt --output-dir /kaggle/working/runs/transunet_crop512_pw5/predictions \
    --batch-size 1 --workers 4 --tile 512 --overlap 0.25 --tile-batch 8